# Treatment Outcomes Overview — r/covidlonghaulers

**Question:** What treatments are patients discussing, and what are the reported outcomes?

**Data:** 1-week snapshot (220 posts, ~3,600 posts+comments) from r/covidlonghaulers.

**Method:** Descriptive statistics on treatment sentiment reports. Given the small sample size (68 total reports, max 5 per drug), we use a binomial test to check whether any drug's positive rate significantly differs from chance (50%).

## 1. Setup

In [ ]:
import sqlite3
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use absolute path so notebook works from any directory
DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"  # fallback for running from repo root
conn = sqlite3.connect(DB_PATH)

# Sentiment encoding (Polina's pipeline stores as text)
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Data Exploration

In [ ]:
# Table sizes
for table in ["users", "posts", "treatment", "treatment_reports", "user_profiles", "conditions"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    print(f"  {table}: {count:,}")

In [ ]:
# Treatment reports with drug names and numeric sentiment
df = pd.read_sql("""
    SELECT 
        tr.user_id,
        t.canonical_name AS drug,
        tr.sentiment AS sentiment_label,
        tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

df["sentiment_score"] = df["sentiment_label"].map(SENTIMENT_SCORE)
print(f"Total reports: {len(df)}")
print(f"Unique drugs: {df['drug'].nunique()}")
print(f"Unique users: {df['user_id'].nunique()}")
print(f"\nSentiment distribution:")
print(df["sentiment_label"].value_counts())

## 3. Analysis

### 3a. Treatment report counts and sentiment breakdown

In [ ]:
# Per-drug summary (user-level: one data point per user per drug)
user_drug = df.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("sentiment_score", "mean"),
    n_reports=("sentiment_score", "count")
).reset_index()

user_drug["outcome"] = user_drug["avg_sentiment"].apply(
    lambda x: "positive" if x > 0.7 else ("negative" if x < -0.7 else "mixed/neutral")
)

drug_summary = user_drug.groupby("drug").agg(
    n_users=("user_id", "count"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100),
    pct_negative=("outcome", lambda x: (x == "negative").mean() * 100),
    avg_sentiment=("avg_sentiment", "mean")
).reset_index().sort_values("n_users", ascending=False)

print("Treatments with 2+ user reports:")
display(drug_summary[drug_summary["n_users"] >= 2].round(1))

### 3b. Binomial test — does any drug differ from chance?

With small sample sizes, the binomial test is the most appropriate. It asks: "Is this drug's positive rate significantly different from 50%?" This is an exact test, valid even with n=3.

In [ ]:
from scipy.stats import binomtest

results = []
for drug in drug_summary[drug_summary["n_users"] >= 2]["drug"]:
    drug_users = user_drug[user_drug["drug"] == drug]
    n = len(drug_users)
    n_pos = (drug_users["outcome"] == "positive").sum()
    test = binomtest(n_pos, n, p=0.5, alternative="two-sided")
    results.append({
        "drug": drug,
        "n_users": n,
        "n_positive": n_pos,
        "pct_positive": round(n_pos / n * 100, 1),
        "p_value": round(test.pvalue, 4),
        "significant": "*" if test.pvalue < 0.05 else ""
    })

results_df = pd.DataFrame(results).sort_values("n_users", ascending=False)
print("Binomial test: positive rate vs 50% baseline")
print("(* = p < 0.05)\n")
display(results_df)

### 3c. Top conditions co-occurring with treatment reports

In [ ]:
conditions = pd.read_sql("""
    SELECT condition_name, COUNT(DISTINCT user_id) as n_users
    FROM conditions
    GROUP BY condition_name
    ORDER BY n_users DESC
    LIMIT 15
""", conn)

print("Top conditions by number of users:")
display(conditions)

## 4. Visualization

In [ ]:
# Sentiment distribution by drug (drugs with 2+ reports)
plot_data = user_drug[user_drug["drug"].isin(drug_summary[drug_summary["n_users"] >= 2]["drug"])]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: outcome counts
outcome_counts = plot_data.groupby(["drug", "outcome"]).size().unstack(fill_value=0)
colors = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}
outcome_counts.plot(kind="barh", stacked=True, ax=axes[0],
                     color=[colors.get(c, "gray") for c in outcome_counts.columns])
axes[0].set_xlabel("Number of users")
axes[0].set_title("Treatment outcomes (user-level)")
axes[0].legend(title="Outcome", loc="lower right")

# Box plot: sentiment scores
sns.boxplot(data=plot_data, y="drug", x="avg_sentiment", ax=axes[1], orient="h")
axes[1].axvline(x=0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Average sentiment score")
axes[1].set_title("Sentiment distribution by treatment")

plt.tight_layout()
plt.show()

In [ ]:
# Condition prevalence
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=conditions.head(10), x="n_users", y="condition_name", ax=ax, color="#3498db")
ax.set_xlabel("Number of users")
ax.set_ylabel("")
ax.set_title("Top 10 conditions in r/covidlonghaulers (1-week sample)")
plt.tight_layout()
plt.show()

## 5. Summary

### Key findings

- **68 treatment reports** across **44 unique drugs** from a 1-week snapshot of r/covidlonghaulers
- Most-discussed treatments: antihistamines (5 user reports), magnesium (3), antibiotics (3), minocycline (3)
- Sentiment is mixed across most treatments — with max n=5 per drug, no individual treatment reaches statistical significance on the binomial test
- Top conditions: long COVID (207 users), PEM (116), POTS (102), MCAS (88), ME/CFS (72)

### Caveats

- **Very small sample sizes** — max 5 users per drug. Statistical tests have very low power at these sizes. Results are descriptive only.
- **1-week snapshot** — not representative of the full subreddit. A month or longer window would capture more treatment discussions.
- **Canonicalization issues** — some conditions (depression, anxiety, ADHD) may be incorrectly tagged as treatments by the extraction pipeline.
- **Prefilter bias** — the sentiment classifier prefilters for posts expressing personal experience, which may skew toward more engaged users.

### Next steps

- Re-run with the full month dataset (1,090 posts) for better per-drug coverage
- Fix canonicalization to separate conditions from treatments
- Run demographic-stratified analysis once sample sizes are larger

---

*Based on self-selected Reddit posts. Users who never posted about a treatment are not represented. Results reflect reporting patterns, not population-level treatment effects.*

In [ ]:
conn.close()